Treino das regressões logísticas

São treinados três modelos por cenário:

1. `susc` — suscetibilidade isolada;
2. `ssr_abs` — SSR absoluto isolado;
3. `combined` — suscetibilidade e SSR absoluto.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

sys.path.append("/code/scripts")

from seasonal_utils import train_logistic_model

In [ ]:
# ALTERAR APENAS ESTA VARIÁVEL

area = "centro"


samples = Path(
    f"/code/data/processed/{area}/model_inputs/seasonal/"
    "samples_train_2008_2024.csv"
)

scenario_susc = {
    "C7": "susc_lr",
    "C8": "susc_rf"
}

print("Área:", area)
print("Amostra de treino:", samples)

In [ ]:
train = pd.read_csv(samples)

burned_ssr = train.loc[
    train["burned"] == 1,
    "ssr_abs"
]

unburned_ssr = train.loc[
    train["burned"] == 0,
    "ssr_abs"
]

mann_whitney = mannwhitneyu(
    burned_ssr,
    unburned_ssr,
    alternative="two-sided"
)

mann_whitney_df = pd.DataFrame([{
    "variable": "ssr_abs",
    "u_statistic": mann_whitney.statistic,
    "p_value": mann_whitney.pvalue,
    "burned_mean": burned_ssr.mean(),
    "unburned_mean": unburned_ssr.mean(),
    "burned_median": burned_ssr.median(),
    "unburned_median": unburned_ssr.median()
}])

mann_whitney_df

In [ ]:
for scenario, susceptibility in scenario_susc.items():
    model_defs = {
        "susc": [susceptibility],
        "ssr_abs": ["ssr_abs"],
        "combined": [susceptibility, "ssr_abs"]
    }

    base = Path(
        f"/code/data/results/{area}/{scenario}/seasonal"
    )
    model_dir = base / "models"
    model_dir.mkdir(parents=True, exist_ok=True)

    coefficient_rows = []

    for model_name, features in model_defs.items():
        model_file = model_dir / f"logit_{model_name}.joblib"

        model = train_logistic_model(
            table=train,
            features=features,
            target="burned",
            output_model=model_file
        )

        coefficient_rows.append({
            "scenario": scenario,
            "model": model_name,
            "term": "intercept",
            "coefficient": model.intercept_[0],
            "odds_ratio": np.exp(model.intercept_[0])
        })

        for feature, coefficient in zip(
            features,
            model.coef_[0]
        ):
            coefficient_rows.append({
                "scenario": scenario,
                "model": model_name,
                "term": feature,
                "coefficient": coefficient,
                "odds_ratio": np.exp(coefficient)
            })

    coefficients = pd.DataFrame(coefficient_rows)

    results = base / "logistic_models.xlsx"

    with pd.ExcelWriter(results) as writer:
        coefficients.to_excel(
            writer,
            sheet_name="coefficients",
            index=False
        )

        mann_whitney_df.to_excel(
            writer,
            sheet_name="mann_whitney_ssr",
            index=False
        )

        train.describe().to_excel(
            writer,
            sheet_name="sample_describe"
        )

    print()
    print("Cenário:", scenario)
    display(coefficients)
    print("Resultados:", results)